In [1]:
from textgrid import TextGrid
import os
import pandas as pd

import statistics
from collections import defaultdict
from collections import Counter
import csv
import re

In [4]:
df = pd.read_csv("../../../Desktop/Uni/phd/Статья 1/EKSKFK_speakers.txt", sep="\t")
print(df.head())

      ID gender  birth_year   birth_place        lat        lon  \
0  001_N      N        1970     Põltsamaa  58.655203  25.976880   
1  002_N      N        1976         Tartu  58.381034  26.719553   
2  003_M      M        1980       Tallinn  59.437324  24.745230   
3  004_N      N        1979         Tartu  58.381034  26.719553   
4  005_N      N        1984  Järva-Madise  59.116070  25.656133   

                       lived_in    profession education     L1  ...  \
0               Põltsama, Tartu  akadeemiline      kõrg  eesti  ...   
1                         Tartu     teenindus      kõrg  eesti  ...   
2  Tallinn. Tartu (alates 1998)  akadeemiline      kõrg  eesti  ...   
3                  Rapla, Tartu  akadeemiline      kõrg  eesti  ...   
4    Järva-Madise, Paide, Tartu  akadeemiline      kõrg  eesti  ...   

  mother_homeplace father_L1 father_homeplace  ankeedi_kp handedness  \
0              NaN       NaN              NaN  09.11.2006        NaN   
1              NaN       N

In [5]:
df2 = df[["ID", "gender", "birth_year", "ankeedi_kp", "education"]]
df2["ankeedi_year"] = df2["ankeedi_kp"].apply(
    lambda x: x.split('.')[2].split('/')[0]
)
df2["age"] = df2["ankeedi_year"].astype(int) - df2["birth_year"].astype(int)
print(df2.head())

      ID gender  birth_year  ankeedi_kp education ankeedi_year  age
0  001_N      N        1970  09.11.2006      kõrg         2006   36
1  002_N      N        1976  09.11.2006      kõrg         2006   30
2  003_M      M        1980  09.11.2006      kõrg         2006   26
3  004_N      N        1979  09.11.2006      kõrg         2006   27
4  005_N      N        1984  13.11.2006      kõrg         2006   22


/var/folders/6g/70z4bwk11l771tqx2cthhf6h0000gn/T/ipykernel_2074/2289807065.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["ankeedi_year"] = df2["ankeedi_kp"].apply(
/var/folders/6g/70z4bwk11l771tqx2cthhf6h0000gn/T/ipykernel_2074/2289807065.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["age"] = df2["ankeedi_year"].astype(int) - df2["birth_year"].astype(int)


In [6]:
df_final = df2[["ID", "gender", "ankeedi_year", "age", "education"]]
print(df_final)
df_final.to_csv('speakers.csv')

        ID gender ankeedi_year  age education
0    001_N      N         2006   36      kõrg
1    002_N      N         2006   30      kõrg
2    003_M      M         2006   26      kõrg
3    004_N      N         2006   27      kõrg
4    005_N      N         2006   22      kõrg
..     ...    ...          ...  ...       ...
202  235_M      M         2021   30      kõrg
203  236_M      M         2021   34      kesk
204  237_M      M         2021   31      kesk
205  238_M      M         2022   41      kõrg
206  239_M      M         2022   46      kõrg

[207 rows x 5 columns]


In [7]:
file_paths = ['../../../Desktop/Uni/phd/Статья 1/SKK0_TG', '../../../Desktop/Uni/phd/Статья 1/SKK2_TG']

rows = []
sounds = []

# quick access to participant data
info_map = (df_final
            .set_index("ID")[["ankeedi_year", "age", "education"]]
            .to_dict("index"))

def overlaps(a_min, a_max, b_min, b_max):
    return (b_min < a_max) and (b_max > a_min)

def collect_morphs_for_words(words_tier, morph_tier):
   
    word_to_morphs = {i: [] for i, w in enumerate(words_tier) if w.mark.strip()}
    word_items = [(i, w.minTime, w.maxTime) for i, w in enumerate(words_tier) if w.mark.strip()]

    # need to sort it out.
    word_items.sort(key=lambda x: x[1])
    morph_items = [(m.minTime, m.maxTime, m.mark.strip()) for m in morph_tier if m.mark.strip()]
    morph_items.sort(key=lambda x: x[0])

    j = 0
    for i, w_min, w_max in word_items:
        # promote morphemes that definitely end before the word
        while j < len(morph_items) and morph_items[j][1] <= w_min:
            j += 1

        k = j
        # collect all morphemes that may intersect with the word
        while k < len(morph_items) and morph_items[k][0] < w_max:
            m_min, m_max, m_mark = morph_items[k]
            if overlaps(w_min, w_max, m_min, m_max):
                word_to_morphs[i].append(m_mark)
            k += 1

    return word_to_morphs

n_fp = len(file_paths)
for i_fp, file_path in enumerate(file_paths):
    print(f"Processing {i_fp + 1}/{n_fp} file path ({(i_fp + 1) / n_fp * 100:.1f}%)")

    condition = 0 if file_path == 'SKK0_TG' else 1

    files = [f for f in os.listdir(file_path) if f.endswith(".TextGrid")]
    n_f = len(files)

    for i_f, filename in enumerate(files):
        print(f"Processing {i_f + 1}/{n_f} file ({(i_f + 1) / n_f * 100:.1f}%)")
        file = os.path.join(file_path, filename)

        try:
            tg = TextGrid.fromFile(file)
        except Exception as e:
            print("Failed:", file, e)
            continue

        base = filename.replace(".TextGrid", "")      # "SKK001-003_M"
        left, gender_letter = base.split("_")         # left="SKK001-003"
        dialogue_id = left.replace('SKK', '')
        id_number = left.split("-")[-1]               # "003"
        participant_id = f"{id_number}_{gender_letter}"
        gender = 0 if gender_letter == "N" else 1

        info = info_map.get(participant_id)
        if info is None:
            #  Nones
            continue
        year = info["ankeedi_year"]
        age = info["age"]
        education = info["education"]

        words_tier = next(t for t in tg.tiers if t.name == "sõnad")
        morph_tier = next(t for t in tg.tiers if t.name == "morf")

        # collect morphemes for all words ONCE
        word_to_morphs = collect_morphs_for_words(words_tier, morph_tier)

        # real_word_indices Omce
        real_word_indices = [i for i, w in enumerate(words_tier)
                             if w.mark.strip() and word_to_morphs.get(i)]
      
        idx_to_pos = {idx: pos for pos, idx in enumerate(real_word_indices)}  # O(1)

        for i, w in enumerate(words_tier):
            word = w.mark.strip()
            if not word:
                continue

            morphs = word_to_morphs.get(i, [])
            morph = " | ".join(morphs)

            if not morph:
                continue

            pos = idx_to_pos.get(i)
            if pos is None:
                # in case of problems - skip the context
                continue

            left_idxs = real_word_indices[max(0, pos - 5):pos]
            right_idxs = real_word_indices[pos + 1:pos + 6]

            prev_words = [words_tier[j].mark.strip() for j in left_idxs]
            next_words = [words_tier[j].mark.strip() for j in right_idxs]
            context = " ".join(prev_words + [word] + next_words)

            rows.append({
                "conditions": condition,
                "word": word,
                "morph": morph,
                "context": context,
                "dialogue": dialogue_id,
                "ID": participant_id,
                "gender": gender,
                "year": year,
                "age": age,
                "education": education
            })


Processing 1/2 file path (50.0%)
Processing 1/144 file (0.7%)
Processing 2/144 file (1.4%)
Processing 3/144 file (2.1%)
Processing 4/144 file (2.8%)
Processing 5/144 file (3.5%)
Processing 6/144 file (4.2%)
Processing 7/144 file (4.9%)
Processing 8/144 file (5.6%)
Processing 9/144 file (6.2%)
Processing 10/144 file (6.9%)
Processing 11/144 file (7.6%)
Processing 12/144 file (8.3%)
Processing 13/144 file (9.0%)
Processing 14/144 file (9.7%)
Processing 15/144 file (10.4%)
Processing 16/144 file (11.1%)
Processing 17/144 file (11.8%)
Processing 18/144 file (12.5%)
Processing 19/144 file (13.2%)
Processing 20/144 file (13.9%)
Processing 21/144 file (14.6%)
Processing 22/144 file (15.3%)
Processing 23/144 file (16.0%)
Processing 24/144 file (16.7%)
Processing 25/144 file (17.4%)
Processing 26/144 file (18.1%)
Processing 27/144 file (18.8%)
Processing 28/144 file (19.4%)
Processing 29/144 file (20.1%)
Processing 30/144 file (20.8%)
Processing 31/144 file (21.5%)
Processing 32/144 file (22.2%

In [8]:
df_rows = pd.DataFrame(rows)
print(df_rows)
df_sounds = pd.DataFrame(sounds)
#print(df_sounds)

        conditions           word  \
0                1            mis   
1                1             me   
2                1             ka   
3                1             no   
4                1            vot   
...            ...            ...   
483237           1  vaevalisemalt   
483238           1          vaata   
483239           1           ede-   
483240           1             s-   
483241           1            jah   

                                               morph  \
0       mis+0 //_P_ pl n, //    mis+0 //_P_ sg n, //   
1                              mina+0 //_P_ pl n, //   
2                                      ka+0 //_D_ //   
3                                      no+0 //_I_ //   
4                                     vot+0 //_I_ //   
...                                              ...   
483237                     vaeva=lisemalt+0 //_D_ //   
483238                           vaata+0 //_V_ o, //   
483239                          ede+0 //_S_ sg g, /

In [113]:
df_sounds.to_csv("sounds_from_corpus.csv", index=False, encoding="utf-8")

In [36]:
df_rows.to_csv("all_words_from_corpus.csv", index=False, encoding="utf-8")

In [9]:
unique_by_gender = {}
for row in rows:
    gender = row["gender"]
    speaker = row["ID"]

    if gender not in unique_by_gender:
        unique_by_gender[gender] = set()

    unique_by_gender[gender].add(speaker)

for gender, speakers in unique_by_gender.items():
    print(gender, len(speakers))

0 78
1 74


In [10]:
ages = [row["age"] for row in rows if row["age"] is not None]

min_age = min(ages)
max_age = max(ages)

print(min_age)
print(max_age)

20
85


In [11]:
median_age = statistics.median(ages)
print(median_age)

36.0


In [12]:
years = {row["year"] for row in rows if row["year"] is not None}

print(sorted(years))

['2006', '2007', '2007 ', '2008', '2009', '2011', '2012', '2012 ', '2013', '2014', '2015', '2016', '2018', '2019', '2020', '2021', '2022']


In [13]:
education = {row["education"] for row in rows if row["education"] is not None}
print(education)

{'lõpetamata kõrg', 'põhiharidus, kutseharidus', 'kesk', nan, 'kõrg', 'kõrgharidus omandamisel', 'kesk-eri'}


In [14]:
dialogue_genders = defaultdict(set)

for row in rows:
    dialogue = str(row["dialogue"])[:3]  
    gender = row["gender"]
    dialogue_genders[dialogue].add(gender)


mm_count = 0
ff_count = 0
mf_count = 0

for dial, genders in dialogue_genders.items():
    if genders == {1}:
        mm_count += 1
    elif genders == {0}:
        ff_count += 1
    else:
        mf_count += 1

print("M–M:", mm_count)
print("F–F:", ff_count)
print("M–F:", mf_count)

M–M: 22
F–F: 31
M–F: 38


In [15]:
dialogues = {str(row["dialogue"])[:3] for row in rows}
total_dialogues = len(dialogues)

print(total_dialogues)

91


In [16]:
unique_ids = {row["ID"] for row in rows}
count_unique_ids = len(unique_ids)

print("Unique ID:", count_unique_ids)

Unique ID: 152


In [17]:

dialogue_ages = defaultdict(dict)

for row in rows:
    dialog = str(row["dialogue"])[:3] 
    speaker = row["ID"]
    age = row["age"]

    if age is not None:
        dialogue_ages[dialog][speaker] = age


age_differences = []

for dialog, id_age_map in dialogue_ages.items():
    ages = list(id_age_map.values())
    if len(ages) >= 2:  
        diff = max(ages) - min(ages)
        age_differences.append(diff)


min_diff = min(age_differences)
max_diff = max(age_differences)
median_diff = statistics.median(age_differences)

print("Minimaalne vanusevahe:", min_diff)
print("Maksimaalne vanusevahe:", max_diff)
print("Mediaan:", median_diff)


Minimaalne vanusevahe: 0
Maksimaalne vanusevahe: 56
Mediaan: 7


In [18]:
dialogue_ages = defaultdict(dict)

for row in rows:
    dialog = str(row["dialogue"])[:3]
    speaker = row["ID"]
    age = row["age"]
    if age is not None:
        dialogue_ages[dialog][speaker] = age


dialogue_to_diff = {}

for dialog, id_age_map in dialogue_ages.items():
    ages = list(id_age_map.values())
    if len(ages) >= 2:
        diff = max(ages) - min(ages)
    else:
        diff = None  
    dialogue_to_diff[dialog] = diff

for row in rows:
    dialog = str(row["dialogue"])[:3]
    row["age_diff"] = dialogue_to_diff.get(dialog)

In [19]:
# age
dialogue_ages = defaultdict(dict)

for row in rows:
    dialog = str(row["dialogue"])[:3]
    speaker = row["ID"]
    age = row["age"]
    if age is not None:
        dialogue_ages[dialog][speaker] = age


#  age difference
dialogue_to_age_diff = {}

for dialog, id_age_map in dialogue_ages.items():
    ages = list(id_age_map.values())
    if len(ages) >= 2:
        dialogue_to_age_diff[dialog] = max(ages) - min(ages)
    else:
        dialogue_to_age_diff[dialog] = None


# genders
dialogue_genders = defaultdict(set)

for row in rows:
    dialog = str(row["dialogue"])[:3]
    gender = row["gender"]
    if gender is not None:
        dialogue_genders[dialog].add(gender)


#  genders in a dialogue
dialogue_to_gender_type = {}

for dialog, genders in dialogue_genders.items():
    if genders == {0} or genders == {1}:
        dialogue_to_gender_type[dialog] = 0   # same gender
    elif genders == {0, 1}:
        dialogue_to_gender_type[dialog] = 1   # different genders
    else:
        dialogue_to_gender_type[dialog] = None



for row in rows:
    dialog = str(row["dialogue"])[:3]
    row["age_diff"] = dialogue_to_age_diff.get(dialog)
    row["gender_mix"] = dialogue_to_gender_type.get(dialog)


In [20]:
df_rows = pd.DataFrame(rows)
print(df_rows)

        conditions           word  \
0                1            mis   
1                1             me   
2                1             ka   
3                1             no   
4                1            vot   
...            ...            ...   
483237           1  vaevalisemalt   
483238           1          vaata   
483239           1           ede-   
483240           1             s-   
483241           1            jah   

                                               morph  \
0       mis+0 //_P_ pl n, //    mis+0 //_P_ sg n, //   
1                              mina+0 //_P_ pl n, //   
2                                      ka+0 //_D_ //   
3                                      no+0 //_I_ //   
4                                     vot+0 //_I_ //   
...                                              ...   
483237                     vaeva=lisemalt+0 //_D_ //   
483238                           vaata+0 //_V_ o, //   
483239                          ede+0 //_S_ sg g, /

In [21]:
particles = {}

skips = ['issand', 'jeerum', 'issake', 'issakene', 'kurat', 'raisk', 'vabandust', 'aitäh', 'hops', 'kõll', 'pläraki', 'plaks', 'ata', 'tss',
        'vaat', 'mis_asja', 'oot', 'hahahah', 'hahah', 'hahaha', 'hah', 'hahaa', 'hahaah', 'hahahahaa',
        'ahah', 'ahaha', 'ahahah', 'ahahahah',
        'ähäh', 'äh',
        'hee', 'heh', 'hehe', 'hehee', 'eheh', 'ehehe',
        'hihihii', 'hih', 'hii', 'ihi',
        'hohoho', 'hohoo', 'hohoh', 'hohohoo',
        'huhuhuu', 'huh', 'huhh',
        'uhoh', 'uhuhuu',
        'õhõh', 'hõh', 'õõh', 'õõhõ', 'uh', 'uhuhuu', 'uhoh',
        'oeh', 'ohh', 'oh','puh', 'puhh']

for row in rows:
    if row['morph'].endswith('//_I_ //'):
        particle, morph = row['morph'].split('+')
        if particle.startswith(tuple(skips)):
            pass
        elif particle == 'no_jaa':
            particles['no_jah'] = particles.get('no_jah', 0) + 1
        elif particle.startswith('eiei'):
            particles['eiei'] = particles.get('eiei', 0) + 1
        elif particle.startswith('mh') or particle.startswith('hm') or particle.startswith('mmh'):
            particles['mhmh'] = particles.get('mhmh', 0) + 1
        else:
            particles[particle] = particles.get(particle, 0) + 1
print(particles)

{'no': 4742, 'vot': 428, 'mnjaa': 17, 'noh': 337, 'no_jah': 785, 'jajah': 347, 'aeh': 1, 'jah': 274, 'mhmh': 3879, 'uu': 6, 'oi': 144, 'oijah': 9, 'aa': 1087, 'okei': 771, 'oo': 45, 'noo': 36, 'ai': 56, 'eh': 6, 'ahaa': 36, 'voh': 6, 'eiei': 35, 'halloo': 3, 'oioi': 4, 'eia': 1, 'ae': 10, 'soh': 4, 'hoi': 1, 'oih': 10, 'ahhaa': 16, 'tst': 1, 'näh': 5, 'haa': 8, 'aah': 7, 'ahha': 4, 'õiu': 1, 'hallo': 4, 'mnjah': 12, 'oioioi': 4, 'juhhei': 1, 'täh': 2, 'ououou': 1, 'ooh': 1, 'aha': 16, 'ena': 1, 'ahh': 1, 'aih': 1, 'häh': 3, 'hei': 2, 'tohoh': 5, 'ehh': 1, 'aiaiai': 2, 'aiaiaiai': 1, 'tjah': 3, 'aiai': 1, 'höh': 1, 'soo': 2, 'ah_jaa': 2, 'ooohoh': 1, 'fui': 1, 'hähh': 1, 'hää': 1, 'oio': 1, 'õhõ': 1, 'mõhh': 1, 'ahaah': 1, 'neh': 1, 'eeh': 1, 'äöä': 1}


In [22]:
for item in rows:
    item['dialogue'] = item['dialogue'].split('-')[0]

In [23]:
counts = Counter((item['dialogue'], item['ID']) for item in rows)
for item in rows:
    item['speaker_word_count'] = counts[(item['dialogue'], item['ID'])]

In [29]:
conditional_forms = [
    '//_V_ ks, //', '//_V_ ksid, //', '//_V_ ksime, //',
    '//_V_ ksite, //', '//_V_ ksin, //'
]

with open("conditional.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["word", "context"])  # to 

    for row in rows:
        if row['morph'].endswith(tuple(conditional_forms)):
            if row['morph'].startswith('taht') == False:
                writer.writerow([row['word'], row['context']])
               # print(row['word'], row['morph'], row['context'])

In [25]:
not_annotated_particles = [
    'tegelikult', 'kindlasti', 'tõesti', 'kahtlemata', 'muidugi', 'üsna', 'nii', 'ütleme',
    'umbes', "väga", "täiesti", "eriti", "õnneks", "kahjuks", "just", "nüd", "sis",
    "ju", "kule", "jumal", 'vist', 'äkki', 'võibolla', 'tõenäoliselt', 'ilmselt'
]

output_dir = "particles"
os.makedirs(output_dir, exist_ok=True)

def safe_filename(text: str) -> str:
    # Remove characters that may break file names
    text = text.strip()
    text = re.sub(r'[\\/*?:"<>|]+', "_", text)
    return text

counts = {}

# rows = [...]  # list of dicts like: {'word': ..., 'context': ..., 'morph': ...}
for row in rows:
    word = row.get("word")
    if word in not_annotated_particles:
        counts[word] = counts.get(word, 0) + 1

        file_path = os.path.join(output_dir, f"{safe_filename(word)}.csv")
        file_exists = os.path.exists(file_path)

        with open(file_path, "a", encoding="utf-8", newline="") as f:
            writer = csv.writer(f, delimiter="\t")
            if not file_exists:
                writer.writerow(["word", "morph", "context"])  # header written once
            writer.writerow([
                word,
                row.get("morph", ""),
                row.get("context", "")
            ])

# Optional: print "jumal" occurrences to console
#for row in rows:
    #if row.get("word") == "jumal":
       # print(row.get("word"), row.get("morph"), row.get("context"))

#print("Done. Counts:", counts)

In [27]:
particle_counts = Counter()
jumal = Counter()
kurat = Counter()
vabandust = Counter()
aitah = Counter()
mis_asi = Counter()
vaat = Counter()
oot = Counter()
eitus = Counter()
vot = Counter()
no = Counter()
noh = Counter()
jah = Counter()
okei = Counter()
noo = Counter()
mhmh = Counter()

polite_words = Counter()

kp_markers = Counter()

arvama = Counter()
uskuma = Counter()
kartma = Counter()
motlema = Counter()
utlema = Counter()

muidugi = Counter()
kahjuks = Counter()


conditional_forms = ['//_V_ ks, //', '//_V_ ksid, //', '//_V_ ksime, //', '//_V_ ksite, //', '//_V_ ksin, //']
conditional = Counter()
imperative = Counter()

teietama = Counter()
sinatama = Counter()

for i, row in enumerate(rows):
    if row['word'] in ['arvan', 'usun', 'eeldan', 'kardan', 'mõtlen', 'kaldun', 'ütlen']:
        kp_markers[(row['dialogue'], row['ID'])] += 1

    if row['word'] == 'arvan':
        arvama[(row['dialogue'], row['ID'])] += 1
    if row['word'] == 'usun':
        uskuma[(row['dialogue'], row['ID'])] += 1
    if row['word'] == 'kardan':
        kartma[(row['dialogue'], row['ID'])] += 1
    if row['word'] == 'mõtlen':
        motlema[(row['dialogue'], row['ID'])] += 1
    if row['word'] == 'ütlen':
        utlema[(row['dialogue'], row['ID'])] += 1

    if row['word'] == 'muidugi':
        muidugi[(row['dialogue'], row['ID'])] += 1
    if row['word'] == 'kahjuks':
        kahjuks[(row['dialogue'], row['ID'])] += 1
        
    if row['word'] in ['palun', 'aitäh', 'vabandust', 'vabandage']:
        polite_words[(row['dialogue'], row['ID'])] += 1
        
    if row['morph'].endswith('//_I_ //'):
        particle, morph = row['morph'].split('+')
        
        if particle not in ['hops', 'kõll', 'pläraki', 'plaks', 'ata', 'tss', 'hahahah', 'hahah', 'hahaha', 'hah',
                            'hahaa', 'hahaah', 'hahahahaa',
                            'ahah', 'ahaha', 'ahahah', 'ahahahah',
                            'ähäh', 'äh',
                            'hee', 'heh', 'hehe', 'hehee', 'eheh', 'ehehe',
                            'hihihii', 'hih', 'hii', 'ihi',
                            'hohoho', 'hohoo', 'hohoh', 'hohohoo',
                            'huhuhuu', 'huh', 'huhh',
                            'uhoh', 'uhuhuu',
                            'õhõh', 'hõh', 'õõh', 'õõhõ', 'uh', 'uhuhuu', 'uhoh',
                            'oeh', 'ohh', 'oh', 'puh', 'puhh']: 
            particle_counts[(row['dialogue'], row['ID'])] += 1
        
       
        if particle in ['issand', 'jeerum', 'issake', 'issakene']:
            jumal[(row['dialogue'], row['ID'])] += 1
        elif particle == 'kurat' or particle == 'raisk':
            kurat[(row['dialogue'], row['ID'])] += 1
        elif particle == 'vabandust':
            vabandust[(row['dialogue'], row['ID'])] += 1
        elif particle == 'aitäh':
            aitah[(row['dialogue'], row['ID'])] += 1
        elif particle == 'mis_asja':
            mis_asi[(row['dialogue'], row['ID'])] += 1
        elif particle == 'vaat':
            vaat[(row['dialogue'], row['ID'])] += 1
        elif particle == 'oot':
            oot[(row['dialogue'], row['ID'])] += 1
        elif particle.startswith('ei'):
            eitus[(row['dialogue'], row['ID'])] += 1
        elif particle.endswith('jah') or particle.endswith('jaa'):
            jah[(row['dialogue'], row['ID'])] += 1
        elif particle == 'no':
            no[(row['dialogue'], row['ID'])] += 1
        elif particle == 'noh':
            noh[(row['dialogue'], row['ID'])] += 1
        elif particle == 'vot':
            vot[(row['dialogue'], row['ID'])] += 1
        elif particle == 'okei':
            okei[(row['dialogue'], row['ID'])] += 1
        elif particle == 'noo':
            okei[(row['dialogue'], row['ID'])] += 1
        elif particle == 'mhmh':
            mhmh[(row['dialogue'], row['ID'])] += 1
            
    if row['morph'].endswith(tuple(conditional_forms)):
        conditional[(row['dialogue'], row['ID'])] += 1

    if row['morph'].endswith('//_V_ ge, //'):
        teietama[(row['dialogue'], row['ID'])] += 1
        if row['morph'] != 'vabandage':
            imperative[(row['dialogue'], row['ID'])] += 1
    if row['morph'].endswith('//_V_ o, //') and rows[i-1]['word'] != 'ei':
        imperative[(row['dialogue'], row['ID'])] += 1
        sinatama[(row['dialogue'], row['ID'])] += 1

    if row['morph'] == 'sina+0 //_P_ pl n, //':
        teietama[(row['dialogue'], row['ID'])] += 1
    elif row['morph'] == 'sina+0 //_P_ sg n, //':
        sinatama[(row['dialogue'], row['ID'])] += 1
    
new_data = {}

for row in rows:
    key = (row['dialogue'], row['ID'])

    if key not in new_data:
        new_data[key] = {
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'conditions': row['conditions'],
            'gender': row['gender'],
            'year': row['year'],
            'age': row['age'],
            'education': row['education'],
            'age_diff': row['age_diff'],
            'speaker_word_count': row['speaker_word_count'],
            'polite_words': polite_words[key],
            'particle': particle_counts[key],
            'jumal': jumal[key],
            'kurat': kurat[key],
            'vabandust': vabandust[key],
            'aitah': aitah[key],
            'mis_asi': mis_asi[key],
            'vaat': vaat[key],
            'oot': oot[key],
            'eitus': eitus[key],
            'vot': vot[key], 
            'no': no[key], 
            'noh': noh[key], 
            'jah': jah[key],
            'okei': okei[key], 
            'noo': noo[key], 
            'mhmh': mhmh[key],
            'kp_markers': kp_markers[key],
            'arvama': arvama[key],
            'uskuma': uskuma[key],
            'kartma': kartma[key],
            'motlema': motlema[key],
            'utlema': utlema[key],
            'muidugi': muidugi[key],
            'kahjuks': kahjuks[key],
            'conditional': conditional[key],
            'imperative': imperative[key],
            'tietama': teietama[key],
            'sinatama':sinatama[key]
            
        }


result = list(new_data.values())
print(result[0])

{'dialogue': '042', 'ID': '124_N', 'conditions': 1, 'gender': 0, 'year': '2015', 'age': 78, 'education': 'kõrg', 'age_diff': 36, 'speaker_word_count': 5519, 'polite_words': 0, 'particle': 58, 'jumal': 0, 'kurat': 0, 'vabandust': 0, 'aitah': 0, 'mis_asi': 0, 'vaat': 1, 'oot': 0, 'eitus': 0, 'vot': 1, 'no': 14, 'noh': 14, 'jah': 27, 'okei': 0, 'noo': 0, 'mhmh': 0, 'kp_markers': 0, 'arvama': 0, 'uskuma': 0, 'kartma': 0, 'motlema': 0, 'utlema': 0, 'muidugi': 18, 'kahjuks': 1, 'conditional': 15, 'imperative': 30, 'tietama': 12, 'sinatama': 38}


In [28]:
df = pd.DataFrame(result)
df.to_csv('dialogue_speakers.csv', index=False, encoding='utf-8')